### Bienvenido a la Semana 5 Día 1

AutoGen AgentChat!

Esto debería parecer simple y familiar, porque tiene mucho en común con Crew y OpenAI Agents SDK

In [2]:
# Cargar variables de entorno desde el archivo .env
from dotenv import load_dotenv
load_dotenv(override=True)

True

### Primer concepto: el Modelo

In [4]:
# Importar el cliente de modelo OpenAI y crear una instancia para GPT-4o-mini
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [3]:
# Importar el cliente de modelo Ollama y crear una instancia para llama3.2
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2")

### Segundo concepto: El Mensaje

In [21]:
# Importar TextMessage y crear un mensaje de ejemplo
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="Me gustaría ir a San José", source="user")
message

TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 4, 17, 31, 40, 211104, tzinfo=datetime.timezone.utc), content='Me gustaría ir a San José', type='TextMessage')

### Tercer concepto: El Agente

In [11]:
# Importar AssistantAgent y crear un agente de aerolínea con modelo y mensaje de sistema
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="Eres un asistente muy servicial de una aerolínea. Das respuestas breves y divertidas.",
    model_client_stream=True
)

### Juntarlo todo con on_messages

In [12]:
# Importar CancellationToken y enviar el mensaje al agente para obtener respuesta
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'¡Excelente elección! San José es como un café bien preparado: una mezcla perfecta de cultura, aventura y buena vibra. ☕✈️ ¿Ya tienes pensado cuándo te gustaría volar?'

### Vamos a crear una base de datos local de precios de boletos

In [13]:
# Importar os y sqlite3, eliminar base de datos existente si existe, crear tabla de ciudades
import os
import sqlite3

# Eliminar archivo de base de datos existente si existe
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

# Crear la base de datos y la tabla
conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [14]:
# Función para guardar precio de ciudad, poblar base de datos con algunas ciudades
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

# Algunas ciudades!
save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)
save_city_price("San José", 1200)

In [15]:
# Función para obtener precio de boleto de ida y vuelta a una ciudad
def get_city_price(city_name: str) -> float | None:
    """ Obtener el precio del boleto de ida y vuelta para viajar a la ciudad """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

In [17]:
# Probar la función get_city_price con Roma
get_city_price("San josé")

1200.0

In [19]:
# Crear un agente inteligente de aerolínea con herramienta para obtener precios
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=model_client,
    system_message="Eres un asistente muy servicial de una aerolínea. Das respuestas breves y divertidas, como el precio de un billete de ida y vuelta.",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [22]:
# Enviar mensaje al agente inteligente y mostrar mensajes internos y respuesta
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_1e0FKLRDWWe8JsPFb5vhTND7', arguments='{"city_name":"San José"}', name='get_city_price')]
[FunctionExecutionResult(content='1200.0', name='get_city_price', call_id='call_1e0FKLRDWWe8JsPFb5vhTND7', is_error=False)]


'¡Perfecto! Un viaje a San José te costará alrededor de $1200 para un billete de ida y vuelta. ¡Eso es solo un café y un par de piñas coladas en la playa! ☕🍍✈️ ¡A planear la aventura!'